Cell 1 — Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

BASE_DIR = Path(r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting")
ADVANCED_DIR = BASE_DIR / "data" / "advanced"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
SUBMISSION_DIR = BASE_DIR / "submissions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

Cell 2 — Load data

In [2]:
train = pd.read_csv(ADVANCED_DIR / "train_advanced.csv")
test = pd.read_csv(PROCESSED_DIR / "test_features.csv")

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

print(train.shape)
print(test.shape)
print(test["date"].min(), test["date"].max())

(3000888, 29)
(28512, 20)
2017-08-16 00:00:00 2017-08-31 00:00:00


Cell 3 — Feature list

In [3]:
target = "sales"

features = [
    "store_nbr", "family", "onpromotion",
    "year", "month", "day", "quarter", "week", "day_of_week", "is_weekend",
    "city", "state", "type", "cluster",
    "dcoilwtico", "is_payday", "is_national_holiday", "earthquake_period",
    "sales_lag_7", "sales_lag_14", "sales_lag_28",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_28"
]

Cell 4 — Train final model

In [4]:
X = train[features]
y = np.log1p(train[target])

X = pd.get_dummies(X, drop_first=True)

final_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

train_columns = X.columns

print("Model trained.")
print(len(train_columns))

Model trained.
94


Cell 5 — Prepare history

In [5]:
history = train[
    [
        "date", "store_nbr", "family", "sales"
    ]
].copy()

predictions = []

Cell 6 — Recursive prediction loop

In [6]:
for current_date in sorted(test["date"].unique()):

    print("Predicting:", current_date)

    current_test = test[test["date"] == current_date].copy()

    latest = (
        history.sort_values(["store_nbr", "family", "date"])
        .groupby(["store_nbr", "family"])
        .tail(28)
    )

    lag_table = (
        latest
        .groupby(["store_nbr", "family"])
        .agg(
            sales_lag_7=("sales", lambda x: x.iloc[-7] if len(x) >= 7 else 0),
            sales_lag_14=("sales", lambda x: x.iloc[-14] if len(x) >= 14 else 0),
            sales_lag_28=("sales", lambda x: x.iloc[-28] if len(x) >= 28 else 0),
            rolling_mean_7=("sales", lambda x: x.iloc[-7:].mean() if len(x) >= 7 else 0),
            rolling_mean_14=("sales", lambda x: x.iloc[-14:].mean() if len(x) >= 14 else 0),
            rolling_mean_28=("sales", lambda x: x.iloc[-28:].mean() if len(x) >= 28 else 0),
            rolling_std_7=("sales", lambda x: x.iloc[-7:].std() if len(x) >= 7 else 0),
            rolling_std_28=("sales", lambda x: x.iloc[-28:].std() if len(x) >= 28 else 0)
        )
        .reset_index()
    )

    current_test = current_test.merge(
        lag_table,
        on=["store_nbr", "family"],
        how="left"
    )

    lag_features = [
        "sales_lag_7", "sales_lag_14", "sales_lag_28",
        "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
        "rolling_std_7", "rolling_std_28"
    ]

    current_test[lag_features] = current_test[lag_features].fillna(0)

    X_current = current_test[features]
    X_current = pd.get_dummies(X_current, drop_first=True)
    X_current = X_current.reindex(columns=train_columns, fill_value=0)

    log_preds = final_model.predict(X_current)
    preds = np.expm1(log_preds)
    preds = np.maximum(preds, 0)

    current_test["sales"] = preds

    predictions.append(
        current_test[["id", "sales"]]
    )

    new_history = current_test[
        ["date", "store_nbr", "family", "sales"]
    ]

    history = pd.concat(
        [history, new_history],
        ignore_index=True
    )

Predicting: 2017-08-16 00:00:00
Predicting: 2017-08-17 00:00:00
Predicting: 2017-08-18 00:00:00
Predicting: 2017-08-19 00:00:00
Predicting: 2017-08-20 00:00:00
Predicting: 2017-08-21 00:00:00
Predicting: 2017-08-22 00:00:00
Predicting: 2017-08-23 00:00:00
Predicting: 2017-08-24 00:00:00
Predicting: 2017-08-25 00:00:00
Predicting: 2017-08-26 00:00:00
Predicting: 2017-08-27 00:00:00
Predicting: 2017-08-28 00:00:00
Predicting: 2017-08-29 00:00:00
Predicting: 2017-08-30 00:00:00
Predicting: 2017-08-31 00:00:00


Cell 7 — Save recursive submission

In [7]:
submission_recursive = pd.concat(
    predictions,
    ignore_index=True
)

submission_recursive = submission_recursive.sort_values("id")

submission_path = SUBMISSION_DIR / "submission_recursive_xgboost_lags.csv"

submission_recursive.to_csv(
    submission_path,
    index=False
)

print("Saved:", submission_path)
print(submission_recursive.shape)
submission_recursive.head()

Saved: C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting\submissions\submission_recursive_xgboost_lags.csv
(28512, 2)


,id,sales
0,3000888,4.055172
1,3000889,0.059201
2,3000890,4.649125
3,3000891,2104.148193
4,3000892,0.073674


Cell 8

In [8]:
old_sub = pd.read_csv(
    SUBMISSION_DIR / "submission_final_xgboost_lags.csv"
)

new_sub = pd.read_csv(
    SUBMISSION_DIR / "submission_recursive_xgboost_lags.csv"
)

difference = (
    old_sub["sales"] -
    new_sub["sales"]
).abs()

print("Max difference:", difference.max())
print("Mean difference:", difference.mean())
print("Rows changed:", (difference > 1e-8).sum())

Max difference: 2636.3359999999993
Mean difference: 45.88100590135688
Rows changed: 25198
